# 03 Feature Engineering

This notebook compiles a clean, case-level feature table for **Problem 6: The Overpayment Signal**.

### Objectives:
1. Aggregate individual transaction history from `payments.csv` to case level.
2. Merge transaction aggregations with raw case attributes from `cases.csv` into a single table with exactly one row per `case_id`.
3. Engineer features representing payment behavior, frequency/patterns, temporal aspects, and potential overpayment signals (e.g. post-closure payments, payment-to-award ratios).
4. Display data properties (shape, schema, null values, descriptive statistics) and save the table to `data/processed/case_features.csv`.

---

## 1. Import Required Libraries

We import standard packages and add the source directory to the python path to load our reusable module `src/feature_engineering.py`.

In [1]:
import pandas as pd
import numpy as np
import sys
import os

# Add src to path if running inside notebooks directory
sys.path.append(os.path.abspath('..'))

from src.feature_engineering import build_case_features
print('Libraries and build_case_features imported successfully!')

Libraries and build_case_features imported successfully!


## 2. Execute Feature Engineering Pipeline

We invoke the modular pipeline `build_case_features` to construct our case-level feature table. The logic reads from the raw data files, aggregates transactions, and joins them with case characteristics.

In [2]:
cases_path = '../data/raw/cases.csv'
payments_path = '../data/raw/payments.csv'

df_features = build_case_features(cases_path, payments_path)
print('Feature table successfully compiled!')

Feature table successfully compiled!


## 3. Verify and Display Feature Table Properties

We run checks to confirm the table's integrity (uniqueness of case_id, missing values, column types) and print descriptive summaries.

### A. Feature Table Shape

In [3]:
print(f'Feature table shape: {df_features.shape[0]} rows (cases), {df_features.shape[1]} columns')

Feature table shape: 4200 rows (cases), 34 columns


### B. Engineered Columns List

In [4]:
print(list(df_features.columns))

['case_id', 'status', 'district', 'age_band', 'language_preference', 'tenure', 'household_size', 'opened_date', 'closure_month', 'monthly_award', 'opened_year', 'opened_month', 'case_age_months', 'contact_attempts', 'months_since_review', 'payment_adjustments', 'total_payments', 'total_amount_paid', 'avg_payment_amount', 'min_payment_amount', 'max_payment_amount', 'std_payment_amount', 'distinct_payment_months', 'max_payments_in_single_month', 'multi_payment_months_count', 'has_same_month_multi_payments', 'post_closure_payment_count', 'is_post_closure_paid', 'total_post_closure_amount', 'total_excess_amount', 'avg_payment_to_award_ratio', 'max_payment_to_award_ratio', 'num_adjusted_payments', 'adjustment_rate']


### C. Missing-Value Summary

In [5]:
null_counts = df_features.isnull().sum()
print('Columns with missing values:')
print(null_counts[null_counts > 0])
print('\n(All other columns have 0 missing values.)')

Columns with missing values:
closure_month    3949
dtype: int64

(All other columns have 0 missing values.)


### D. Basic Descriptive Statistics

We inspect the distribution, scale, and ranges of the numerical features.

In [6]:
display(df_features.describe())

,household_size,monthly_award,opened_year,opened_month,case_age_months,contact_attempts,months_since_review,payment_adjustments,total_payments,total_amount_paid,...,multi_payment_months_count,has_same_month_multi_payments,post_closure_payment_count,is_post_closure_paid,total_post_closure_amount,total_excess_amount,avg_payment_to_award_ratio,max_payment_to_award_ratio,num_adjusted_payments,adjustment_rate
count,4200.000000,4200.000000,4200.000000,4200.000000,4200.000000,4200.000000,4200.000000,4200.000000,4200.000000,4200.000000,...,4200.000000,4200.000000,4200.000000,4200.000000,4200.000000,4200.000000,4200.000000,4200.000000,4200.000000,4200.000000
mean,2.469286,824.029710,2022.833333,6.352857,31.647143,2.351190,11.504762,0.969048,5.894286,4791.382733,...,0.014762,0.010952,0.043333,0.014286,32.910198,97.559600,0.987510,1.048457,0.932381,0.157627
std,1.365903,279.527184,1.344755,3.401352,16.062179,1.687619,6.830962,0.985967,0.671444,1757.529421,...,0.155156,0.104091,0.372332,0.118680,304.251676,380.431735,0.092717,0.101924,1.114412,0.188710
min,1.000000,372.150000,2021.000000,1.000000,4.000000,0.000000,0.000000,0.000000,1.000000,379.310000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.749527,0.749527,0.000000,0.000000
25%,1.000000,612.580000,2022.000000,3.000000,18.000000,1.000000,6.000000,0.000000,6.000000,3526.410000,...,0.000000,0.000000,0.000000,0.000000,0.000000,32.057500,0.961642,1.031433,0.000000,0.000000
50%,2.000000,765.420000,2023.000000,6.000000,32.000000,2.000000,11.000000,1.000000,6.000000,4473.625000,...,0.000000,0.000000,0.000000,0.000000,0.000000,57.675000,0.986082,1.045402,1.000000,0.166667
75%,3.000000,1000.122500,2024.000000,9.000000,46.000000,3.000000,17.000000,2.000000,6.000000,5861.015000,...,0.000000,0.000000,0.000000,0.000000,0.000000,90.812500,1.002597,1.053788,2.000000,0.333333
max,6.000000,1850.800000,2025.000000,12.000000,59.000000,9.000000,23.000000,6.000000,10.000000,15128.200000,...,4.000000,1.000000,4.000000,1.000000,5581.700000,7504.560000,2.675611,2.962703,5.000000,1.000000


### E. Unique Case Key Confirmation

In [7]:
is_unique = not df_features['case_id'].duplicated().any()
print(f'Is case_id guaranteed to be unique (exactly one row per case)? {is_unique}')
print(f'Total cases in raw cases: {len(pd.read_csv(cases_path))}')
print(f'Total cases in engineered table: {len(df_features)}')

Is case_id guaranteed to be unique (exactly one row per case)? True
Total cases in raw cases: 4200
Total cases in engineered table: 4200


### F. Sample Records

We display the first 5 records of our case-level feature table.

In [8]:
display(df_features.head())

,case_id,status,district,age_band,language_preference,tenure,household_size,opened_date,closure_month,monthly_award,...,multi_payment_months_count,has_same_month_multi_payments,post_closure_payment_count,is_post_closure_paid,total_post_closure_amount,total_excess_amount,avg_payment_to_award_ratio,max_payment_to_award_ratio,num_adjusted_payments,adjustment_rate
0,C-30000,Active,Calder Central,45-59,English,Private tenancy,2,2024-11-13,NaN,1026.76,...,0,0,0,0,0.0,37.24,0.940785,1.036269,0,0.000000
1,C-30001,Active,Weybridge,30-44,English,Private tenancy,1,2023-10-21,NaN,497.45,...,0,0,0,0,0.0,61.49,1.009137,1.056327,0,0.000000
2,C-30002,Active,Northgate,45-59,English,No fixed abode,2,2025-02-02,NaN,531.45,...,0,0,0,0,0.0,26.35,0.974613,1.022730,0,0.000000
3,C-30003,Active,Weybridge,60-74,Spanish,Owner-occupier,2,2024-10-06,NaN,926.43,...,0,0,0,0,0.0,56.09,0.978673,1.038017,1,0.166667
4,C-30004,Active,Weybridge,30-44,English,No fixed abode,1,2023-03-19,NaN,514.36,...,0,0,0,0,0.0,84.22,1.000813,1.055117,1,0.166667


## 4. Save Processed Feature Table

We save the final table to the `data/processed/` folder for downstream model training and prioritization.

In [9]:
output_path = '../data/processed/case_features.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_features.to_csv(output_path, index=False)
print(f'Feature table successfully written to {output_path}!')

Feature table successfully written to ../data/processed/case_features.csv!


## Feature Engineering Summary

* **Payment Behaviors**: Aggregated transaction statistics (total, average, standard deviation) establishing standard baseline payment patterns.
* **Frequency Patterns**: Captured same-month duplicate disbursements to isolate double-disbursement risks.
* **Potential Signals**: Quantified post-closure payments, excess amount discrepancies, and adjustment rates to represent primary potential anomaly indicators.
* **Case Characteristics**: Merged review gap months and contact attempts to proxy administrative risks.
* **Demographics**: Preserved raw demographic features (district, age band, language preference, tenure) for post-modeling fairness checks.


## Potential Concerns

* **Collinearity**: High correlations exist between totals, payment counts, and monthly awards. While handled well by tree-based prioritization models, linear models would require feature selection.
* **Information Leakage**: Post-closure features rely on the historical closure month. These are highly informative for auditing past leakage but would be zero for active cases.
* **Fairness & Governance**: Demographic fields are excluded from the predictive model inputs to prevent encoding algorithmic bias and are reserved solely for post-modeling auditing.
